# Minimum squared Euclidean distance

`d2_min` sets the high-SNR slope. Read straight from the simulator.

In [ ]:
import json, subprocess, pathlib
import numpy as np
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent
MSPRS = ROOT / "build" / "bin" / "msprs"
FIGURES = ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

plt.rcParams.update({
    "font.family": "sans-serif", "font.size": 8, "axes.labelsize": 8,
    "axes.titlesize": 8, "legend.fontsize": 7, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "axes.linewidth": 0.6, "lines.linewidth": 1.2,
    "axes.grid": True, "grid.alpha": 0.3, "grid.linestyle": ":",
    "figure.dpi": 110, "savefig.dpi": 200, "savefig.bbox": "tight",
    "pdf.fonttype": 42, "ps.fonttype": 42,
})

def run(*args):
    """One call to the simulator."""
    return subprocess.run([str(MSPRS), *map(str, args)],
                          capture_output=True, text=True, check=True).stdout


In [ ]:
rows = [l.split() for l in run("--mode", "bounds").splitlines() if not l.startswith("#")]
d2 = {}
for fam, L0, _, es5, gain in rows:
    d2.setdefault(fam, []).append((int(L0), float(es5), float(gain)))
for fam, v in d2.items():
    print(fam, [(L, round(e, 3), round(g, 2)) for L, e, g in sorted(v)])

In [ ]:
ASK4 = 4.0
fig, ax = plt.subplots(figsize=(4.6, 3.4))
ax.axhline(ASK4, color="#d62728", ls="--", lw=1.4, label="4-ASK")
for fam, colour, marker in (("unbalanced", "#000000", "o"), ("balanced", "#404040", "s")):
    v = sorted(d2[fam])
    ax.plot([L for L, _, _ in v], [e for _, e, _ in v],
            color=colour, marker=marker, ms=5, lw=1.6, label=f"MS-PRS {fam}")

ax.set(xlabel="$L_0$", ylabel=r"$d^2_{\min}$", xticks=[3, 4, 5, 6], ylim=(3.4, 9.6))
ax.secondary_yaxis("right",
    functions=(lambda y: 10 * np.log10(np.maximum(y, 1e-9) / ASK4),
               lambda g: ASK4 * 10 ** (g / 10))).set_ylabel("gain over 4-ASK (dB)")
ax.legend(loc="upper left")
for ext in ("pdf", "png"):
    fig.savefig(FIGURES / f"msed_gain.{ext}")